# Notebook 2: Hugging Face Computer Vision for Workshop
## OCR, Object Detection, and Segmentation

This notebook is designed for teaching sessions.

Each model is loaded and tested in its own step so students can clearly follow the workflow.

### Learning Goals

1. Run OCR, detection, and segmentation with Hugging Face models.
2. Understand how changing model size changes speed and output detail.
3. Practice reading and explaining model outputs in class discussions.

### Session Flow

1. Part A: OCR (`trocr-small` -> `trocr-base`)
2. Part B: Object Detection (`yolos-tiny` -> `detr-resnet-50`)
3. Part C: Segmentation (`segformer-b0` -> `segformer-b2`)

In [ ]:
%pip install -q -U transformers accelerate torch pillow requests matplotlib

In [ ]:
import time
import torch
import numpy as np
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display
from transformers import pipeline, TrOCRProcessor, VisionEncoderDecoderModel

print('CUDA available:', torch.cuda.is_available())

## Part A: OCR (Image to Text)

We start with printed-text OCR and compare a smaller and larger TrOCR checkpoint.

Class focus:
- Accuracy and readability of extracted text.
- Relative loading and generation speed on local machines.

### Step A1: Smaller OCR Model

Model: `microsoft/trocr-small-printed` (~61M)

This model is lightweight and good for quick local demos.

In [ ]:
ocr_image_url = 'https://fki.tic.heia-fr.ch/static/img/a01-122-02-00.jpg'
ocr_image = Image.open(requests.get(ocr_image_url, stream=True, timeout=30).raw).convert('RGB')
display(ocr_image)

Parameters we highlight here:

- `max_new_tokens`: upper bound on generated text length.
- `skip_special_tokens=True`: cleaner decoded text output.

In [ ]:
trocr_small_id = 'microsoft/trocr-small-printed'

start = time.time()
trocr_small_processor = TrOCRProcessor.from_pretrained(trocr_small_id)
trocr_small_model = VisionEncoderDecoderModel.from_pretrained(trocr_small_id)
print('Loaded model:', trocr_small_id)
print(f'Load time: {time.time() - start:.2f} sec')

pixel_values_small = trocr_small_processor(images=ocr_image, return_tensors='pt').pixel_values
generated_ids_small = trocr_small_model.generate(pixel_values_small, max_new_tokens=64)
ocr_text_small = trocr_small_processor.batch_decode(generated_ids_small, skip_special_tokens=True)[0]

print('OCR output (small):')
print(ocr_text_small)

### Step A2: Larger OCR Model

Model: `microsoft/trocr-base-printed` (~300M)

Expected classroom observation:
- Often better at difficult characters than the small model.
- Usually slower to load and run.

In [ ]:
trocr_base_id = 'microsoft/trocr-base-printed'

start = time.time()
trocr_base_processor = TrOCRProcessor.from_pretrained(trocr_base_id)
trocr_base_model = VisionEncoderDecoderModel.from_pretrained(trocr_base_id)
print('Loaded model:', trocr_base_id)
print(f'Load time: {time.time() - start:.2f} sec')

pixel_values_base = trocr_base_processor(images=ocr_image, return_tensors='pt').pixel_values
generated_ids_base = trocr_base_model.generate(pixel_values_base, max_new_tokens=64)
ocr_text_base = trocr_base_processor.batch_decode(generated_ids_base, skip_special_tokens=True)[0]

print('OCR output (base):')
print(ocr_text_base)

### OCR Reflection

Discuss with students:
- Which result is cleaner?
- Does the larger model justify the extra runtime for your use case?

## Part B: Object Detection

Now we compare a tiny detector and a larger detector on the same image.

Focus points:
- Number and confidence of detections.
- Speed and stability of bounding boxes.

### Step B1: Tiny Detector

Model: `hustvl/yolos-tiny` (~6.5M)

Highlighted parameter:
- Detection threshold (`score >= 0.70`) to remove weak predictions.

In [ ]:
det_image_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
det_image = Image.open(requests.get(det_image_url, stream=True, timeout=30).raw).convert('RGB')
display(det_image)

In [ ]:
yolos_tiny_id = 'hustvl/yolos-tiny'

start = time.time()
if torch.cuda.is_available():
    yolos_tiny_detector = pipeline(
        'object-detection',
        model=yolos_tiny_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    yolos_tiny_detector = pipeline('object-detection', model=yolos_tiny_id)

print('Loaded model:', yolos_tiny_id)
print(f'Load time: {time.time() - start:.2f} sec')

In [ ]:
yolos_tiny_predictions = yolos_tiny_detector(det_image)
yolos_tiny_predictions = [p for p in yolos_tiny_predictions if p['score'] >= 0.70][:10]

_, ax = plt.subplots(figsize=(10, 6))
ax.imshow(det_image)
ax.axis('off')
ax.set_title('YOLOS Tiny Detections')

for pred in yolos_tiny_predictions:
    box = pred['box']
    x = box['xmin']
    y = box['ymin']
    w = box['xmax'] - box['xmin']
    h = box['ymax'] - box['ymin']

    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 4, f"{pred['label']} {pred['score']:.2f}", color='yellow', fontsize=9, backgroundcolor='black')

plt.show()

### Step B2: Larger Detector

Model: `facebook/detr-resnet-50` (~41.6M)

Expected classroom observation:
- Often stronger object coverage.
- Usually more compute than YOLOS tiny.

In [ ]:
detr_model_id = 'facebook/detr-resnet-50'

start = time.time()
if torch.cuda.is_available():
    detr_detector = pipeline(
        'object-detection',
        model=detr_model_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    detr_detector = pipeline('object-detection', model=detr_model_id)

print('Loaded model:', detr_model_id)
print(f'Load time: {time.time() - start:.2f} sec')

In [ ]:
detr_predictions = detr_detector(det_image)
detr_predictions = [p for p in detr_predictions if p['score'] >= 0.70][:10]

_, ax = plt.subplots(figsize=(10, 6))
ax.imshow(det_image)
ax.axis('off')
ax.set_title('DETR ResNet-50 Detections')

for pred in detr_predictions:
    box = pred['box']
    x = box['xmin']
    y = box['ymin']
    w = box['xmax'] - box['xmin']
    h = box['ymax'] - box['ymin']

    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='cyan', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 4, f"{pred['label']} {pred['score']:.2f}", color='white', fontsize=9, backgroundcolor='black')

plt.show()

### Object Detection Reflection

Prompt students to discuss:
- Did DETR find objects YOLOS missed?
- Was the speed difference noticeable on your hardware?

## Part C: Semantic Segmentation

We now assign class labels to pixels.

We compare two SegFormer sizes to discuss detail vs runtime tradeoff.

### Step C1: Smaller Segmentation Model

Model: `nvidia/segformer-b0-finetuned-ade-512-512` (~3.75M)

In [ ]:
seg_image_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
seg_image = Image.open(requests.get(seg_image_url, stream=True, timeout=30).raw).convert('RGB')
display(seg_image)

In [ ]:
segformer_b0_id = 'nvidia/segformer-b0-finetuned-ade-512-512'

start = time.time()
if torch.cuda.is_available():
    segmenter_b0 = pipeline(
        'image-segmentation',
        model=segformer_b0_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    segmenter_b0 = pipeline('image-segmentation', model=segformer_b0_id)

print('Loaded model:', segformer_b0_id)
print(f'Load time: {time.time() - start:.2f} sec')

In [ ]:
b0_segments = segmenter_b0(seg_image)
b0_segments = sorted(b0_segments, key=lambda s: s['score'], reverse=True)

print('Top labels from SegFormer B0:')
for seg in b0_segments[:5]:
    print(f"- {seg['label']}: {seg['score']:.3f}")

top_b0_masks = b0_segments[:3]
_, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(seg_image)
axes[0].set_title('Original')
axes[0].axis('off')

for i, seg in enumerate(top_b0_masks, start=1):
    axes[i].imshow(np.array(seg['mask']), cmap='viridis')
    axes[i].set_title(f"{seg['label']} ({seg['score']:.2f})")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### Step C2: Larger Segmentation Model

Model: `nvidia/segformer-b2-finetuned-ade-512-512` (~27M)

Expected classroom observation:
- Usually richer segmentation quality.
- Usually slower than B0.

In [ ]:
segformer_b2_id = 'nvidia/segformer-b2-finetuned-ade-512-512'

start = time.time()
if torch.cuda.is_available():
    segmenter_b2 = pipeline(
        'image-segmentation',
        model=segformer_b2_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    segmenter_b2 = pipeline('image-segmentation', model=segformer_b2_id)

print('Loaded model:', segformer_b2_id)
print(f'Load time: {time.time() - start:.2f} sec')

In [ ]:
b2_segments = segmenter_b2(seg_image)
b2_segments = sorted(b2_segments, key=lambda s: s['score'], reverse=True)

print('Top labels from SegFormer B2:')
for seg in b2_segments[:5]:
    print(f"- {seg['label']}: {seg['score']:.3f}")

top_b2_masks = b2_segments[:3]
_, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(seg_image)
axes[0].set_title('Original')
axes[0].axis('off')

for i, seg in enumerate(top_b2_masks, start=1):
    axes[i].imshow(np.array(seg['mask']), cmap='viridis')
    axes[i].set_title(f"{seg['label']} ({seg['score']:.2f})")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## End Of Notebook 2: Suggested Class Activities

1. Replace sample images with your own and repeat all steps.
2. Ask students to select one model per task for low-resource deployment.
3. Capture qualitative notes on speed vs output quality (no formal benchmarking).